In [ ]:
from google.colab import files
uploaded = files.upload()

Saving face detect.mp4 to face detect.mp4


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# ---------------- USER INPUT ----------------
save_option = input("Do you want to save the output video? (y/n): ").lower()

# ---------------- PARAMETERS ----------------
KNOWN_WIDTH = 14.0   # Average face width in cm
FOCAL_LENGTH = 600   # ⚠️ Replace with calibrated value for accuracy

# ---------------- LOAD FACE DETECTOR ----------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# ---------------- VIDEO INPUT ----------------
cap = cv2.VideoCapture(r"face detect.mp4")

# Read first frame to set size
ret, frame = cap.read()
if not ret:
    print("Error reading video")
    exit()

# Processing resolution
PROCESS_WIDTH = 640
PROCESS_HEIGHT = 480

# Reset video
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# ---------------- VIDEO WRITER ----------------
if save_option == 'y':
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('output.avi', fourcc, 20.0,
                          (PROCESS_WIDTH, PROCESS_HEIGHT))

# ---------------- DISTANCE FUNCTION ----------------
def find_distance(perceived_width):
    return (KNOWN_WIDTH * FOCAL_LENGTH) / perceived_width

# ---------------- SMOOTHING VARIABLE ----------------
prev_distance = 0

# ---------------- MAIN LOOP ----------------
while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        # Loop video continuously
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    # -------- STEP 1: RESIZE --------
    frame = cv2.resize(frame, (PROCESS_WIDTH, PROCESS_HEIGHT))

    # -------- STEP 2: FACE DETECTION --------
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    if len(faces) > 0:
        # -------- STEP 3: TAKE LARGEST FACE --------
        faces = sorted(faces, key=lambda x: x[2], reverse=True)
        (x, y, w, h) = faces[0]

        # -------- STEP 4: DISTANCE ESTIMATION --------
        distance = find_distance(w)

        # -------- STEP 5: SMOOTHING --------
        distance = 0.7 * prev_distance + 0.3 * distance
        prev_distance = distance

        # -------- DRAW --------
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, f"Distance: {round(distance,2)} cm",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2)

    # -------- STEP 6: DISPLAY --------
    cv2_imshow(frame)

    # -------- STEP 7: SAVE --------
    if save_option == 'y':
        out.write(frame)

    # Exit key
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# ---------------- CLEANUP ----------------
cap.release()
if save_option == 'y':
    out.release()

cv2.destroyAllWindows()